# Collaborative Filtering Benchmark (Simplified)

**Goal:** Compare 4 similarity measures  
(Cosine, Adjusted‑Cosine, Pearson, Jaccard) for both **user‑based** and **item‑based** memory CF on MovieLens *ml‑latest‑small*.

Metrics  
* Rating prediction: **RMSE**, **MAE**  
* Top‑N quality: **Precision**, **Recall**, **F1**

Just run the notebook top‑to‑bottom. Tune parameters in the final cell.


In [21]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, KNNBasic, KNNWithMeans, accuracy, SVD
from surprise.model_selection import train_test_split
from collections import defaultdict
from sklearn.metrics.pairwise import pairwise_distances

# Load data
df = pd.read_csv("ml-latest-small/ratings.csv")
df = df.drop(columns=["timestamp"])
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
print(f"Loaded {len(df):,} ratings from {df['userId'].nunique()} users and {df['movieId'].nunique()} movies")

Loaded 100,836 ratings from 610 users and 9724 movies


In [17]:
# Define evaluation function
def evaluate(predictions, threshold=4.0, top_n=10):
    # RMSE and MAE
    rmse = accuracy.rmse(predictions, verbose=False)
    mae = accuracy.mae(predictions, verbose=False)
    
    # Precision, Recall, F1
    user_relevant = defaultdict(set)
    user_preds = defaultdict(list)
    
    for uid, iid, true_r, pred_r, _ in predictions:
        if true_r >= threshold:
            user_relevant[uid].add(iid)
        user_preds[uid].append((iid, pred_r))
    
    precisions, recalls = [], []
    for uid in user_preds:
        rec_items = [iid for (iid, _) in sorted(user_preds[uid], key=lambda x: x[1], reverse=True)[:top_n]]
        relevant = user_relevant.get(uid, set())
        if len(relevant) == 0:
            continue
        tp = len(set(rec_items) & relevant)
        prec = tp / len(rec_items)
        rec = tp / len(relevant)
        precisions.append(prec)
        recalls.append(rec)
    
    avg_precision = np.mean(precisions) if precisions else 0
    avg_recall = np.mean(recalls) if recalls else 0
    f1 = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) else 0
    return rmse, mae, avg_precision, avg_recall, f1



In [18]:
# Experiment parameters
similarities = [
    {'name': 'cosine', 'user_based': True},
    {'name': 'pearson', 'user_based': True},
    {'name': 'pearson', 'user_based': False},
    {'name': 'cosine', 'user_based': False},  # Adjusted Cosine for item-based
    {'name': 'jaccard', 'user_based': True},
    {'name': 'msd', 'user_based': True},     
    {'name': 'msd', 'user_based': False}
]
k_values = [5, 10, 20, 30, 50, 100, 150]
results = []

In [19]:
# Main experiment loop
for sim in similarities:
    for k in k_values:
        if sim['name'] == 'jaccard':
            # Custom Jaccard similarity
            print("Computing Jaccard similarity...")
            binary_matrix = np.zeros((trainset.n_users, trainset.n_items))
            for u in range(trainset.n_users):
                for (i, _) in trainset.ur[u]:
                    binary_matrix[u, i] = 1
            jaccard_sim = 1 - pairwise_distances(binary_matrix, metric='jaccard')
            
            # Predict using custom Jaccard
            predictions = []
            for uid, iid, true_r in testset:
                try:
                    u_inner = trainset.to_inner_uid(uid)
                    i_inner = trainset.to_inner_iid(iid)
                except ValueError:
                    predictions.append((uid, iid, true_r, None, {}))
                    continue
                
                # Find users who rated the item
                neighbors = []
                for (v_inner, r) in trainset.ir[i_inner]:
                    if v_inner == u_inner:
                        continue
                    sim_score = jaccard_sim[u_inner, v_inner]
                    neighbors.append((sim_score, r))
                
                neighbors.sort(reverse=True, key=lambda x: x[0])
                neighbors = neighbors[:k]
                
                sum_sim = sum_r = 0
                for sim_score, r in neighbors:
                    sum_sim += sim_score
                    sum_r += sim_score * r
                
                pred = trainset.global_mean
                if sum_sim > 0:
                    pred = sum_r / sum_sim
                predictions.append((uid, iid, true_r, pred, {}))
            
            valid_predictions = [p for p in predictions if p[3] is not None]
            rmse, mae, prec, rec, f1 = evaluate(valid_predictions)
        else:
            # Use Surprise algorithms
            if sim['user_based']:
                algo = KNNBasic(k=k, sim_options=sim)
            else:
                algo = KNNWithMeans(k=k, sim_options=sim)
            
            algo.fit(trainset)
            predictions = algo.test(testset)
            rmse, mae, prec, rec, f1 = evaluate(predictions)
        
        results.append({
            'similarity': sim['name'],
            'user_based': sim['user_based'],
            'k': k,
            'RMSE': rmse,
            'MAE': mae,
            'Precision': prec,
            'Recall': rec,
            'F1': f1
        })

# Convert results to DataFrame
results_df = pd.DataFrame(results)
print(results_df)

# Optionally save to CSV
results_df.to_csv('cf_experiment_results.csv', index=False)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Comput

c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Computing Jaccard similarity...


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Computing Jaccard similarity...


...

Computing Jaccard similarity...


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Computing Jaccard similarity...


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

## Sparcity experiment

In [20]:
# Sparse experiment
sparsity_results = []

# Introduce more sparsity: Keep only 50% of the original ratings
sparse_df = df.sample(frac=0.5, random_state=42)
sparse_data = Dataset.load_from_df(sparse_df[['userId', 'movieId', 'rating']], reader)
sparse_trainset, sparse_testset = train_test_split(sparse_data, test_size=0.2, random_state=42)
print(f"Sparse data: {len(sparse_df):,} ratings from {sparse_df['userId'].nunique()} users and {sparse_df['movieId'].nunique()} movies")

for sim in similarities:
    for k in k_values:
        if sim['name'] == 'jaccard':
            print("Computing Jaccard similarity on sparse data...")
            binary_matrix = np.zeros((sparse_trainset.n_users, sparse_trainset.n_items))
            for u in range(sparse_trainset.n_users):
                for (i, _) in sparse_trainset.ur[u]:
                    binary_matrix[u, i] = 1
            jaccard_sim = 1 - pairwise_distances(binary_matrix, metric='jaccard')
            
            predictions = []
            for uid, iid, true_r in sparse_testset:
                try:
                    u_inner = sparse_trainset.to_inner_uid(uid)
                    i_inner = sparse_trainset.to_inner_iid(iid)
                except ValueError:
                    predictions.append((uid, iid, true_r, None, {}))
                    continue
                
                neighbors = []
                for (v_inner, r) in sparse_trainset.ir[i_inner]:
                    if v_inner == u_inner:
                        continue
                    sim_score = jaccard_sim[u_inner, v_inner]
                    neighbors.append((sim_score, r))
                
                neighbors.sort(reverse=True, key=lambda x: x[0])
                neighbors = neighbors[:k]
                
                sum_sim = sum_r = 0
                for sim_score, r in neighbors:
                    sum_sim += sim_score
                    sum_r += sim_score * r
                
                pred = sparse_trainset.global_mean
                if sum_sim > 0:
                    pred = sum_r / sum_sim
                predictions.append((uid, iid, true_r, pred, {}))
            
            valid_predictions = [p for p in predictions if p[3] is not None]
            rmse, mae, prec, rec, f1 = evaluate(valid_predictions)
        else:
            if sim['user_based']:
                algo = KNNBasic(k=k, sim_options=sim)
            else:
                algo = KNNWithMeans(k=k, sim_options=sim)
            
            algo.fit(sparse_trainset)
            predictions = algo.test(sparse_testset)
            rmse, mae, prec, rec, f1 = evaluate(predictions)
        
        sparsity_results.append({
            'similarity': sim['name'],
            'user_based': sim['user_based'],
            'k': k,
            'RMSE': rmse,
            'MAE': mae,
            'Precision': prec,
            'Recall': rec,
            'F1': f1
        })

# Convert to DataFrame and print
sparsity_df = pd.DataFrame(sparsity_results)
print(sparsity_df)

sparsity_df.to_csv('cf_sparcity_experiment_results.csv', index=False)

Sparse data: 50,418 ratings from 610 users and 7529 movies
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson si

c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Computing Jaccard similarity on sparse data...


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Computing Jaccard similarity on sparse data...


...

Computing Jaccard similarity on sparse data...


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Computing Jaccard similarity on sparse data...


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

## Model-based CF

### Normal dataset

In [22]:
n_factors_list = [10, 20, 50, 100, 150]
results = []

for n_factors in n_factors_list:
    algo = SVD(n_factors=n_factors, random_state=42)
    algo.fit(trainset)
    predictions = algo.test(testset)
    rmse, mae, prec, rec, f1 = evaluate(predictions)
    
    results.append({
        'model': 'FunkSVD',
        'n_factors': n_factors,
        'RMSE': rmse,
        'MAE': mae,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# Results DataFrame
results_df = pd.DataFrame(results)
print(results_df)

# Optionally save
results_df.to_csv('funk_model_experiment_results.csv', index=False)

     model  n_factors      RMSE       MAE  Precision    Recall        F1
0  FunkSVD         10  0.875861  0.673401   0.663453  0.680995  0.672110
1  FunkSVD         20  0.878010  0.675098   0.662277  0.680455  0.671243
2  FunkSVD         50  0.877468  0.674175   0.659252  0.678277  0.668629
3  FunkSVD        100  0.880746  0.676573   0.658579  0.677646  0.667977
4  FunkSVD        150  0.884763  0.680084   0.656058  0.674169  0.664991


### Sparcity

In [24]:
n_factors_list = [10, 20, 50, 100, 150]
results = []

for n_factors in n_factors_list:
    algo = SVD(n_factors=n_factors, random_state=42)
    algo.fit(sparse_trainset)
    predictions = algo.test(sparse_testset)
    rmse, mae, prec, rec, f1 = evaluate(predictions)
    
    results.append({
        'model': 'FunkSVD',
        'n_factors': n_factors,
        'RMSE': rmse,
        'MAE': mae,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# Results DataFrame
results_df = pd.DataFrame(results)
print(results_df)

# Optionally save
results_df.to_csv('funk_model_experiment_results.csv', index=False)

     model  n_factors      RMSE       MAE  Precision    Recall        F1
0  FunkSVD         10  0.881666  0.680626   0.679332  0.809167  0.738587
1  FunkSVD         20  0.881533  0.680332   0.676401  0.805650  0.735390
2  FunkSVD         50  0.886531  0.684499   0.673654  0.805777  0.733816
3  FunkSVD        100  0.891173  0.689874   0.675669  0.808254  0.736038
4  FunkSVD        150  0.892077  0.690856   0.672006  0.801701  0.731146
